In [1]:
# ============================================================
# TRANSFORM DRIVERS — LOCAL SILVER LAYER
# ============================================================

import os
from pyspark.sql import functions as F

In [2]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:08:34 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:08:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:08:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:08:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:08:35 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:08:35 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:08:35 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [3]:
# -----------------------------------------
# 2. Define Bronze + Silver paths
# -----------------------------------------
bronze_file = f"{BRONZE_ROOT}/drivers/drivers.parquet"
silver_folder = f"{SILVER_ROOT}/drivers"
silver_file = f"{silver_folder}/drivers_silver.parquet"

os.makedirs(silver_folder, exist_ok=True)

print("Bronze:", bronze_file)
print("Silver:", silver_file)

Bronze: /Users/manoharazalki/F1-ANALYTICS/data/bronze/drivers/drivers.parquet
Silver: /Users/manoharazalki/F1-ANALYTICS/data/silver/drivers/drivers_silver.parquet


In [4]:
# -----------------------------------------
# 3. Read Bronze Drivers
# -----------------------------------------
drivers_df = spark.read.parquet(bronze_file)
print("Bronze drivers loaded")
drivers_df.show(5, truncate=False)

Bronze drivers loaded
+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+------------------------------------------------------------------+
|driverId |name               |dateOfBirth|nationality|url                                           |ingest_timestamp          |source_file                                                       |
+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+------------------------------------------------------------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers

In [5]:
# -----------------------------------------
# 4. Drop URL column
# -----------------------------------------
drivers_dropped_df = drivers_df.drop("url")

In [6]:
# -----------------------------------------
# 5. Standardize + rename columns
# -----------------------------------------
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("dateOfBirth", "date_of_birth")
)

print("Drivers renamed preview")
drivers_renamed_df.show(5, truncate=False)

Drivers renamed preview
+---------+-------------------+-------------+-----------+--------------------------+------------------------------------------------------------------+
|driver_id|name               |date_of_birth|nationality|ingest_timestamp          |source_file                                                       |
+---------+-------------------+-------------+-----------+--------------------------+------------------------------------------------------------------+
|abate    |{carlo, abate}     |1932-07-10   |italian    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|abecassis|{george, abecassis}|1913-03-21   |british    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|acheson  |{kenny, acheson}   |1957-11-27   |british    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|
|adams    |{philippe, adams}  |1969-11-19   |belgian    |2026-06

In [7]:
# -----------------------------------------
# 6. Create driver_name (givenName + familyName)
# -----------------------------------------
drivers_concatenated_df = (
    drivers_renamed_df
        .withColumn(
            "driver_name",
            F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))
        )
        .drop("name")
)

print("✔ Driver name created")
drivers_concatenated_df.show(5, truncate=False)

✔ Driver name created
+---------+-------------+-----------+--------------------------+------------------------------------------------------------------+----------------+
|driver_id|date_of_birth|nationality|ingest_timestamp          |source_file                                                       |driver_name     |
+---------+-------------+-----------+--------------------------+------------------------------------------------------------------+----------------+
|abate    |1932-07-10   |italian    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|carlo abate     |
|abecassis|1913-03-21   |british    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|george abecassis|
|acheson  |1957-11-27   |british    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|kenny acheson   |
|adams    |1969-11-19   |belgian    |2026-06-08 23:07:49.257734|/Users/manoharazalki

In [8]:
# -----------------------------------------
# 7. Remove duplicates
# -----------------------------------------
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

In [9]:
# -----------------------------------------
# 8. Title Case nationality + driver_name
# -----------------------------------------
drivers_final_df = (
    drivers_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))
        .withColumn("driver_name", F.initcap(F.col("driver_name")))
)

print("✔ Drivers Silver preview")
drivers_final_df.show(5, truncate=False)

✔ Drivers Silver preview
+---------+-------------+-----------+--------------------------+------------------------------------------------------------------+----------------+
|driver_id|date_of_birth|nationality|ingest_timestamp          |source_file                                                       |driver_name     |
+---------+-------------+-----------+--------------------------+------------------------------------------------------------------+----------------+
|Cannoc   |1933-06-21   |Canadian   |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|John Cannon     |
|Changy   |1922-02-05   |Belgian    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|Alain De Changy |
|abate    |1932-07-10   |Italian    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|Carlo Abate     |
|abecassis|1913-03-21   |British    |2026-06-08 23:07:49.257734|/Users/manoharaza

In [10]:
# -----------------------------------------
# 9. Write Silver output (Option C)
# -----------------------------------------
(
    drivers_final_df
        .write
        .mode("overwrite")
        .parquet(silver_file)
)

print(f"✔ Drivers Silver written to: {silver_file}")

✔ Drivers Silver written to: /Users/manoharazalki/F1-ANALYTICS/data/silver/drivers/drivers_silver.parquet


In [11]:
# -----------------------------------------
# 10. Read back for validation
# -----------------------------------------
spark.read.parquet(silver_file).show(5, truncate=False)

+---------+-------------+-----------+--------------------------+------------------------------------------------------------------+----------------+
|driver_id|date_of_birth|nationality|ingest_timestamp          |source_file                                                       |driver_name     |
+---------+-------------+-----------+--------------------------+------------------------------------------------------------------+----------------+
|Cannoc   |1933-06-21   |Canadian   |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|John Cannon     |
|Changy   |1922-02-05   |Belgian    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|Alain De Changy |
|abate    |1932-07-10   |Italian    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/drivers.json|Carlo Abate     |
|abecassis|1913-03-21   |British    |2026-06-08 23:07:49.257734|/Users/manoharazalki/F1-ANALYTICS/data/bro